# Feature Evaluation: TF-IDF vs Dense Embeddings

Evaluates text feature extraction methods for tabular ML — comparing classical **TF-IDF** frequency features with **spaCy dense embeddings**. Builds full sklearn pipelines with encoding, scaling, and regression, and benchmarks each approach.

## Key Techniques Covered

1. **TF-IDF Vectorization** - Classical term-frequency features via `TfidfVectorizer`; combined with `PCA` for dimensionality reduction
2. **spaCy Dense Embeddings** - Pre-trained `en_core_web_sm` vectors; each word becomes part of a 96-dim average representation
3. **TextPipeline** (custom transformer) - Concatenates multiple string columns → TF-IDF → dense → PCA, all in one sklearn step
4. **SpacyEmbeddingVectorizer** (custom transformer) - Collapses multiple string columns and maps through spaCy's vector model
5. **ColumnTransformer composition** - Combines numeric imputation, one-hot encoding, target encoding, and text transformers

## Dataset

Uses EPA Fuel Economy data with:
- **Text features**: `eng_dscr`, `trans_dscr`, `model`, `evMotor` (unstructured descriptions)
- **Categorical features**: `make`, `VClass`, `drive`, `fuelType`, `trany`
- **Target**: `city08` (city fuel economy in MPG)

## Key Learnings

- **TF-IDF** captures vocabulary signal well for structured text (engine codes, transmission types); fast and interpretable
- **spaCy embeddings** capture semantic similarity but require model download and are slower
- **Combining text + categorical**: `remainder='passthrough'` in `ColumnTransformer` keeps numeric columns flowing through

**TFIDF**

*Frequency-inverse document frequency (TFIDF)* is a numerical statistic that is intended to reflect how important a word is to a document in a collection or corpus. It is often used as a feature for text classification.

In [ ]:
import sys, pathlib
_vsc_nb = globals().get('__vsc_ipynb_file__')
_nb_dir = pathlib.Path(_vsc_nb).parent if _vsc_nb else pathlib.Path.cwd()
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

import pandas as pd
import numpy as np
from utils import (
    load_fuel_economy_data,
    debug_transformer,
    combine_str_cols_transformer,
    TextPipeline,
    SpacyVectorizer,
    CAT_COLS, LOW_CARDINALITY_COLS, HIGH_CARDINALITY_COLS,
    make_preprocessor,
)

In [ ]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# create X and y
X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# TF-IDF pipeline: preprocessing (TF-IDF text + OHE + target encoding) → regression
pipeline = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('lr', LinearRegression()),
])
pipeline.fit(X_train, y_train)

**Text Embeddings**

*Text Emebeddings* are a type of feature extraction that is used for text data. They are a numerical representation of text that can be used in ML algorithms. They are often used as a feature for text classification. A common example is a vector to represent man, woman, and king. When add the difference between man and woman to king, you get queen.

In [ ]:
!pip install spacy

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:

# Add TFIDF to combination of string columns
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
# replace hashing encoder with target encoder for high cardinality columns

from category_encoders import hashing
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler, OneHotEncoder, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer


# create X and y
X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

# split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# create pipeline fill cylinders with 0 and displ with median

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
displ_imputer = SimpleImputer(strategy='median')
std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
one_hot_encoder = OneHotEncoder(drop='first', max_categories=10, sparse_output=False, handle_unknown='ignore')
hashing_encoder = hashing.HashingEncoder(n_components=10, drop_invariant=True)
target_encoder = TargetEncoder(target_type='continuous', random_state=42)

def debug_transformer(X, name):
    globals()[name] = X
    return X

cat_cols =  ['make', 'model', 'trany', 'drive',
            'VClass', 'eng_dscr', 'evMotor', 'fuelType', 'trans_dscr', ]
low_cardinality_cols = ['VClass', 'drive', 'fuelType', 'trany']
high_cardinality_cols = ['make', 'model', 'eng_dscr', 'evMotor', 'trans_dscr']

# Create the column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cyl_imputer', cylinders_imputer, ['cylinders']),
        ('displ_imputer', displ_imputer, ['displ']),
        ('one_hot_encoder', one_hot_encoder, low_cardinality_cols),
        #('hashing_encoder', hashing_encoder, high_cardinality_cols)
        ('target_encoder', target_encoder, high_cardinality_cols),
        # ('text', TextPipeline(cat_cols), cat_cols)
        ('text', SpacyEmbeddingVectorizer(cat_cols), cat_cols)
    ],
    remainder='passthrough'
)

# pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name': 'tmp_X'})),
    ('std_scaler', std_scaler),
    #('pca', PCA(n_components=10)),
    #  ('minmax_scaler', minmax_scaler, ['range']),
    ('lr', LinearRegression())])

# fit the pipeline
pipeline.fit(X_train.iloc[:1_000], y_train.iloc[:1_000])

In [ ]:
pipeline.score(X_test, y_test)

# Coding Challenge: Predict Mileage from Text Columns

- **Text Vectorization/Embeddings:** When working with unstructured, natural language text (e.g., open-ended descriptions like "fuel-efficient compact car" or concatenated string columns), we must use text vectorization (like TF-IDF) or dense text embeddings to capture semantic meaning and vocabulary frequencies.

- **Categorical Encoding:** If the text columns simply represent structured, discrete categories with a fixed set of possible values (e.g., auto manufacturers like "Toyota" or "Ford", or drive types like "Front-Wheel Drive"), we use Categorical Encoding techniques instead (such as One-Hot Encoding, Target Encoding, or Hashing).


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

cat_cols =  ['make', 'model', 'trany', 'drive',
            'VClass', 'eng_dscr', 'evMotor', 'fuelType', 'trans_dscr', ]


preprocessor = ColumnTransformer(
    transformers = [
        ('text', SpacyEmbeddingVectorizerForChallenge(cat_cols), cat_cols)
    ],
    remainder='drop'
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lr', LinearRegression())
])

pipeline.fit(X_train.iloc[:1_000], y_train.iloc[:1_000])

In [ ]:
pipeline.score(X_test, y_test)